# 03_early_warning.ipynb

Objectives
- Validate **H3**: Can we flag high-risk sellers with enough lead time to act?
- Build a time-aware **(seller, date)** panel with:
  - Daily SLA + GMV metrics.
  - Rolling-window features (7 / 14 / 30 activity days).
  - Future severe-event labels at 7 / 14 / 21 calendar days.
- Train a simple, interpretable baseline model (logistic regression).
- Evaluate:
  - Global metrics: ROC AUC, Average Precision (PR-AUC).
  - Operational metrics: event recall@K, **GMV recall@K** for top-K flagged seller-days.

Notes
- Target = "at least one severe SLA violation in the next H days".
- GMV is used to quantify **GMV at risk captured** by the early-warning model.
- H4 (ROI simulation) will plug into the outputs of this notebook.

## 0. Import & Settings
This section:
- Configures the notebook environment.
- Registers the project root and `src` path.
- Imports:
  - Project-level utilities.
  - Early-warning feature builder (`features.early_warning`).
  - `scikit-learn` for modeling and evaluation.

In [2]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 50)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Import project modules
from config import DATA_INTERIM, DATA_PROCESSED
from features.seller_metrics import validate_orders_sellers
# from features.early_warning import (
#     build_seller_daily_sla,
#     build_rolling_seller_features,
#     label_future_severe_events,
#     prepare_early_warning_panel,
# )
from data.preprocessing import load_orders_sellers

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Fixed RNG for reproducibility
RNG = np.random.RandomState(42)